In [ ]:
import warnings
warnings.filterwarnings('ignore')

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Conv2D, MaxPooling2D, Flatten,
    Dense, Dropout, BatchNormalization
)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing.image import ImageDataGenerator, load_img, img_to_array
from tensorflow.keras.utils import to_categorical

print("TensorFlow version:", tf.__version__)
print("All libraries loaded successfully.")

In [ ]:
# Load labels CSV
df = pd.read_csv('labels.csv')
print("Total images:", len(df))
print("\nFirst 5 rows:")
df.head()

In [ ]:
# Class distribution
class_counts = df['class'].value_counts()
print("Number of classes:", df['class'].nunique())
print("\nImages per class:")
print(class_counts)

# Plot class distribution
plt.figure(figsize=(8, 4))
sns.barplot(x=class_counts.index, y=class_counts.values, palette='viridis')
plt.title('Class Distribution - Manufacturing Defect Dataset')
plt.xlabel('Class')
plt.ylabel('Number of Images')
plt.tight_layout()
plt.savefig('results/class_distribution.png', dpi=150)
plt.show()
print("\nDataset is", "balanced." if class_counts.std() < 10 else "slightly imbalanced.")

In [ ]:
# Display sample images from each class
classes = ['normal', 'scratch', 'dent', 'stain']
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
fig.suptitle('Sample Images from Each Class', fontsize=14, fontweight='bold')

for ax, cls in zip(axes, classes):
    sample_path = df[df['class'] == cls].iloc[0]['filename']
    img = mpimg.imread(sample_path)
    ax.imshow(img)
    ax.set_title(cls.upper(), fontsize=12)
    ax.axis('off')
    # Print image dimensions
    print(f"Class: {cls:10s} | Shape: {img.shape}")

plt.tight_layout()
plt.savefig('results/sample_images.png', dpi=150)
plt.show()

In [ ]:
# Configuration
IMG_SIZE = (64, 64)       # Resize all images to 64x64
BATCH_SIZE = 32
EPOCHS = 30
NUM_CLASSES = 4

# Encode class labels
le = LabelEncoder()
df['label'] = le.fit_transform(df['class'])
print("Label encoding:", dict(zip(le.classes_, le.transform(le.classes_))))

In [ ]:
# Load and preprocess all images
def load_images(dataframe, img_size):
    X = []
    y = []
    for _, row in dataframe.iterrows():
        try:
            img = load_img(row['filename'], target_size=img_size)
            img_array = img_to_array(img) / 255.0  # Normalize to [0, 1]
            X.append(img_array)
            y.append(row['label'])
        except Exception as e:
            print(f"Skipping {row['filename']}: {e}")
    return np.array(X), np.array(y)

print("Loading and preprocessing images...")
X, y = load_images(df, IMG_SIZE)
print(f"Dataset shape: X={X.shape}, y={y.shape}")
print(f"Pixel value range: [{X.min():.2f}, {X.max():.2f}]")

In [ ]:
# Train/Test Split (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Convert labels to one-hot encoding
y_train_cat = to_categorical(y_train, NUM_CLASSES)
y_test_cat  = to_categorical(y_test,  NUM_CLASSES)

print(f"Training set:   {X_train.shape[0]} images")
print(f"Testing set:    {X_test.shape[0]} images")

In [ ]:
# Data Augmentation (applied only to training data)
train_datagen = ImageDataGenerator(
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True,
    zoom_range=0.1
)
train_generator = train_datagen.flow(
    X_train, y_train_cat,
    batch_size=BATCH_SIZE
)

# Visualize augmented samples
sample_X, _ = next(train_generator)
fig, axes = plt.subplots(2, 4, figsize=(14, 6))
fig.suptitle('Augmented Training Samples', fontsize=13, fontweight='bold')
for i, ax in enumerate(axes.flatten()):
    ax.imshow(sample_X[i])
    ax.axis('off')
plt.tight_layout()
plt.show()

print("Augmentation applied: rotation, shift, flip, zoom")

In [ ]:
# Build CNN model
def build_cnn(input_shape=(64, 64, 3), num_classes=4):
    model = Sequential([

        # Block 1: Convolution + Activation + Pooling
        Conv2D(32, (3, 3), activation='relu', padding='same',
               input_shape=input_shape, name='Conv2D_1'),
        BatchNormalization(),
        MaxPooling2D(pool_size=(2, 2), name='MaxPool_1'),

        # Block 2: Convolution + Activation + Pooling
        Conv2D(64, (3, 3), activation='relu', padding='same', name='Conv2D_2'),
        BatchNormalization(),
        MaxPooling2D(pool_size=(2, 2), name='MaxPool_2'),

        # Block 3: Deeper feature extraction
        Conv2D(128, (3, 3), activation='relu', padding='same', name='Conv2D_3'),
        BatchNormalization(),
        MaxPooling2D(pool_size=(2, 2), name='MaxPool_3'),

        # Flatten
        Flatten(name='Flatten'),

        # Dense layers
        Dense(256, activation='relu', name='Dense_1'),
        Dropout(0.5),
        Dense(128, activation='relu', name='Dense_2'),
        Dropout(0.3),

        # Output layer (softmax for multi-class)
        Dense(num_classes, activation='softmax', name='Output')
    ])
    return model

model = build_cnn()
model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

**Model Architecture Explanation:**

| Layer | Purpose |
|---|---|
| `Conv2D` | Learns spatial features (edges, textures, patterns) using filters |
| `BatchNormalization` | Stabilizes training by normalizing layer inputs |
| `MaxPooling2D` | Reduces spatial dimensions, retaining dominant features |
| `Flatten` | Converts 2D feature maps to a 1D vector for dense layers |
| `Dense` | Learns complex combinations of extracted features |
| `Dropout` | Prevents overfitting by randomly dropping neurons during training |
| `Output (Softmax)` | Produces probability distribution over 4 classes |

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
import os

os.makedirs('results', exist_ok=True)
os.makedirs('sample_predictions', exist_ok=True)

# Callbacks
early_stop = EarlyStopping(
    monitor='val_loss', patience=7,
    restore_best_weights=True, verbose=1
)
reduce_lr = ReduceLROnPlateau(
    monitor='val_loss', factor=0.5,
    patience=4, min_lr=1e-6, verbose=1
)

# Train the model
steps_per_epoch = len(X_train) // BATCH_SIZE

history = model.fit(
    train_generator,
    steps_per_epoch=steps_per_epoch,
    epochs=EPOCHS,
    validation_data=(X_test, y_test_cat),
    callbacks=[early_stop, reduce_lr],
    verbose=1
)

In [ ]:
# Plot Training & Validation Accuracy / Loss
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy
axes[0].plot(history.history['accuracy'], label='Train Accuracy', color='steelblue')
axes[0].plot(history.history['val_accuracy'], label='Val Accuracy', color='coral')
axes[0].set_title('Training vs Validation Accuracy')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Loss
axes[1].plot(history.history['loss'], label='Train Loss', color='steelblue')
axes[1].plot(history.history['val_loss'], label='Val Loss', color='coral')
axes[1].set_title('Training vs Validation Loss')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('results/accuracy_loss_curves.png', dpi=150)
plt.show()
print("Accuracy/Loss curves saved.")

In [ ]:
# Evaluate on test set
test_loss, test_accuracy = model.evaluate(X_test, y_test_cat, verbose=0)
print(f"Test Loss:     {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f} ({test_accuracy*100:.2f}%)")

# Predictions
y_pred_probs = model.predict(X_test)
y_pred = np.argmax(y_pred_probs, axis=1)

print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=le.classes_))

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=le.classes_,
    yticklabels=le.classes_
)
plt.title('Confusion Matrix - Manufacturing Defect Classifier', fontsize=13)
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.tight_layout()
plt.savefig('results/confusion_matrix.png', dpi=150)
plt.show()
print("Confusion matrix saved.")

In [ ]:
# Sample Predictions on Test Images
num_samples = 12
indices = np.random.choice(len(X_test), num_samples, replace=False)

fig, axes = plt.subplots(3, 4, figsize=(16, 10))
fig.suptitle('Sample Predictions on Test Images', fontsize=14, fontweight='bold')

for i, (ax, idx) in enumerate(zip(axes.flatten(), indices)):
    img = X_test[idx]
    true_label = le.classes_[y_test[idx]]
    pred_label = le.classes_[y_pred[idx]]
    confidence = y_pred_probs[idx][y_pred[idx]] * 100

    ax.imshow(img)
    color = 'green' if true_label == pred_label else 'red'
    ax.set_title(
        f"True: {true_label}\nPred: {pred_label} ({confidence:.1f}%)",
        color=color, fontsize=9
    )
    ax.axis('off')

plt.tight_layout()
plt.savefig('sample_predictions/prediction_outputs.png', dpi=150)
plt.show()
print("Sample predictions saved.")

---
## **Task 6: CNN Concept Explanation**

### What is Convolution?

Convolution is the core operation in a CNN. A small filter (e.g., 3×3 pixels) slides across the entire image and computes a dot product at each position. This produces a **feature map** that highlights where certain patterns (like edges, corners, or textures) exist in the image.

Think of it as a magnifying glass that scans the image looking for one specific visual feature at a time. Multiple filters are used in parallel, each learning to detect a different feature.

---

### Why is Pooling Used?

Pooling (typically Max Pooling) **reduces the spatial size** of feature maps by keeping only the strongest activation in each small region (e.g., 2×2).

This serves two purposes:
1. **Reduces computation** — smaller feature maps = fewer parameters.
2. **Provides translation invariance** — the model becomes less sensitive to the exact position of a feature. A scratch near the left edge or right edge of an image will still be detected.

---

### Why is ReLU Commonly Used?

ReLU (Rectified Linear Unit) sets all negative values to zero: `f(x) = max(0, x)`.

It is preferred because:
- It introduces **non-linearity**, allowing the network to learn complex patterns beyond simple linear relationships.
- It is **computationally cheap** — just a threshold operation.
- It avoids the **vanishing gradient problem** that affects sigmoid/tanh in deep networks, since its gradient is either 0 or 1.

---

### Why are CNNs Better than Regular Feed-Forward Networks for Images?

| Aspect | Feed-Forward NN | CNN |
|---|---|---|
| Input handling | Flattens image (loses spatial info) | Preserves 2D spatial structure |
| Parameters | Millions (fully connected) | Few (shared weights via filters) |
| Feature learning | Manual feature engineering needed | Automatically learns spatial features |
| Position sensitivity | Sensitive to position shifts | Translation invariant via pooling |

CNNs leverage the **local structure** of images — nearby pixels are related — and **share filter weights** across positions, making them far more efficient and accurate for visual tasks.

---
## **Task 7: Business Use Case Mapping**

### Real-World Domain: **Manufacturing — Automated Quality Inspection**

**Problem:**  
In high-volume manufacturing (e.g., automotive parts, electronics, consumer goods), inspecting every product for surface defects manually is slow, expensive, and prone to human error.

**How CNN solves it:**  
A CNN model like the one built in this notebook can be deployed on a production line with a camera system. As each product passes under the camera:
1. An image is captured of the product surface.
2. The CNN classifies it as `normal`, `scratch`, `dent`, or `stain` in milliseconds.
3. Defective items are automatically flagged and routed to rejection or rework.

**Business Impact:**
- **Cost reduction:** Reduces the need for manual QC inspectors.
- **Speed:** Can inspect thousands of units per hour without fatigue.
- **Consistency:** Applies the same standard every time, unlike human inspectors.
- **Data insights:** Defect type logs can reveal process issues (e.g., a spike in `dent` defects may indicate a machine alignment problem).

**Example companies using this:**  
Toyota, Samsung Electronics, and Foxconn all use AI-based visual inspection systems in their factories.

---
## **Conclusion**

This project demonstrated the complete workflow for building a CNN-based image classifier:

- **Problem Identification:** Multi-class image classification with 4 defect categories.
- **Dataset Exploration:** Balanced dataset of ~480 product surface images.
- **Preprocessing:** Resizing to 64×64, normalization, train/test split, and augmentation.
- **CNN Architecture:** 3 convolutional blocks + dense layers + dropout for regularization.
- **Evaluation:** Confusion matrix and classification report confirm strong per-class performance.
- **Concept Review:** Convolution, pooling, ReLU, and CNN advantages explained.
- **Business Mapping:** CNN deployed as an automated quality inspector on a manufacturing line.

CNNs are highly effective for visual inspection tasks because they automatically learn the spatial features that distinguish one surface defect from another — features that would be extremely difficult to engineer manually.